In [0]:
%run ../00_common/data_utils

In [0]:
def org_field(level, field):
    return F.expr(f"""
        element_at(
            filter(
                from_json(
                    get_json_object(slndc_payload, '$.TouchPoint.OrganizationHierarchyList.OrganizationHierarchy'),
                    'array<struct<Level:string,Code:string,Name:string,Description:string>>'
                ),
                x -> x.Level = '{level}'
            ),
            1
        ).{field}
    """)

In [0]:
def calc_t_touchpoint_stage_df(batch_id):
    bronze_df = spark.read.format("delta").load(get_env_config("bronze_path_touchpoint"))
    latest_batch_df = bronze_df.filter(F.col("slndc_batch_id") == batch_id)

    t_touchpoint_stage_df = latest_batch_df.select(
        # UUID
        F.col("slndc_id").alias("TCPT_ID"),

        # Header
        F.get_json_object("slndc_payload", "$.Header.@Action").alias("TCPT_ACTION"),
        F.get_json_object("slndc_payload", "$.Header.DocumentTimestamp").alias("TCPT_DOCUMENTTIMESTAMP"),
        F.get_json_object("slndc_payload", "$.Header.DocumentUUID").alias("TCPT_DOCUMENTUUID"),

        # TouchPoint root
        F.get_json_object("slndc_payload", "$.TouchPoint.@RecordUUID").alias("TCPT_RECORDUUID"),

        # SourceSystem
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.@Code").alias("TCPT_SourceSystemCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.SourceTimestamp").alias("TCPT_SOURCETIMESTAMP"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.MarketCode").alias("TCPT_MarketCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.AffiliateCode").alias("TCPT_AffiliateCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.DivisionCode").alias("TCPT_DivisionCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.BrandCode").alias("TCPT_BrandCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.TouchPointCode").alias("TCPT_TOUCHPOINTCODE"),
        F.get_json_object("slndc_payload", "$.TouchPoint.SourceSystem.SubMarketCode").alias("TCPT_SubMarketCode"),

        # AuxiliarySourceSystem
        F.get_json_object("slndc_payload", "$.TouchPoint.AuxiliarySourceSystem.@Code").alias("TCPT_AuxiliaryCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.AuxiliarySourceSystem.TouchPointCode").alias("TCPT_AuxiliaryTouchPointCode"),

        # Hygiene
        F.get_json_object("slndc_payload", "$.TouchPoint.Hygiene.HygieneServiceCode").alias("TCPT_HygieneServiceCode"),

        # Attributes —— 包含新增的 5 个字段
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.DistributionChannelCode").alias("TCPT_DistributionChannelCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.TouchPointGroupCode").alias("TCPT_TouchPointGroupCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.RetailerHierarchyCode").alias("TCPT_RetailerHierarchyCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.TouchPointTypeCode").alias("TCPT_TouchPointTypeCode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.EnglishDescription").alias("TCPT_EnglishDescription"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.LocalDescription").alias("TCPT_LocalDescription"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.EnglishFullDescription").alias("TCPT_EnglishFullDescription"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.LocalFullDescription").alias("TCPT_LocalFullDescription"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.Description_en").alias("TCPT_Descriptionen"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.Description_local").alias("TCPT_Descriptionlocal"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.FullDescription_en").alias("TCPT_FullDescriptionen"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.FullDescription_local").alias("TCPT_FullDescriptionlocal"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.URL").alias("TCPT_URL"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.JDECode").alias("TCPT_JDECode"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.Active").alias("TCPT_ACTIVE"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.TouchPointStatus").alias("TCPT_TouchPointStatus"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.OpenDate").alias("TCPT_OpenDate"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.BranchID").alias("TCPT_BranchID"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.RedirectTouchPointCode").alias("TCPT_RedirectTouchPointCode"),
        # >>> 新增 Attributes 字段 <<<
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.Channel").alias("TCPT_Channel"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.CustomerGroup").alias("TCPT_CustomerGroup"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.DTCFlag").alias("TCPT_DTCFlag"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.Region").alias("TCPT_Region"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.City").alias("TCPT_City"),
        F.get_json_object("slndc_payload", "$.TouchPoint.Attributes.CustomAttributeList.CustomAttribute").alias("TCPT_Attr_CustomAttributeList"),
        
        # >>> 新增：存储原始嵌套结构为 JSON 字符串 <<<
        F.get_json_object("slndc_payload", "$.TouchPoint.ContactInformation.PhoneList.Phone").alias("TCPT_PHONELIST"),
        F.get_json_object("slndc_payload", "$.TouchPoint.ContactInformation.AddressList.Address").alias("TCPT_ADDRESSLIST"),
        F.get_json_object("slndc_payload", "$.TouchPoint.CustomAttributeList.CustomAttribute").alias("TCPT_CUSTOMATTRIBUTELIST"),
        F.get_json_object("slndc_payload", "$.TouchPoint.TerminalRegistrationList.TerminalRegistration").alias("TCPT_TERMINALREGISTRATIONLIST"),

        # Organization Hierarchy
        org_field("Global", "Level").alias("TCPT_GLOBAL_Level"),
        org_field("Global", "Code").alias("TCPT_GLOBAL_CODE"),
        org_field("Global", "Name").alias("TCPT_GLOBAL_NAME"),
        org_field("Global", "Description").alias("TCPT_GLOBAL_DESCRIPTION"),
        org_field("Regional", "Level").alias("TCPT_REGIONAL_Level"),
        org_field("Regional", "Code").alias("TCPT_REGIONAL_CODE"),
        org_field("Regional", "Name").alias("TCPT_REGIONAL_NAME"),
        org_field("Regional", "Description").alias("TCPT_REGIONAL_DESCRIPTION"),
        org_field("Affiliate", "Level").alias("TCPT_AFFILIATE_Level"),
        org_field("Affiliate", "Code").alias("TCPT_AFFILIATE_CODE"),
        org_field("Affiliate", "Name").alias("TCPT_AFFILIATE_NAME"),
        org_field("Affiliate", "Description").alias("TCPT_AFFILIATE_DESCRIPTION"),

        # CustomerNumber (位于 TouchPoint 根级)
        F.get_json_object("slndc_payload", "$.TouchPoint.CustomerNumber").alias("TCPT_CustomerNumber"),

        # Audit fields
        F.current_timestamp().alias("TCPT_CREATION_DT"),
        F.lit(TOUCHPOINT_CREATION_UID).alias("TCPT_CREATIONUID"),
        F.current_timestamp().alias("TCPT_UPDATE_DT"),
        F.lit(TOUCHPOINT_UPDATE_UID).alias("TCPT_UPDATEUID"),
        F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
        F.col("slndc_kafka_timestamp").alias("KAFKA_TIMESTAMP")
    )

    stage_table_name = f"{get_env_config('silver_touchpoint_parsed_database')}.t_touchpoint_stage"
    save_to_target_table(t_touchpoint_stage_df, stage_table_name, f"batch_id = '{batch_id}'")

In [0]:
batch_id = dbutils.widgets.get("batch_id")
print(f"batch_id: {batch_id}")

with StepLogger("parse_t_touchpoint_stage", "02", "touchpoint", task_id=batch_id) as logger:
    calc_t_touchpoint_stage_df(batch_id)